# The Price of a Crossing

### Order decides whether the phase bill is payable. Pole placement decides who pays it.

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## Why the access reading changes the question

The phase-closure law has been carrying the term $N$ as "number of stages." Reading it as **access geometry**
renames every term without changing the equation:

$$N\arccos(1/G) + \tau\sqrt{G^2-1} = \Phi$$

- $N$ — the number of interface crossings between the reader and its own returned distinction
- $G$ — how strongly each crossing restores what passes through it
- $\tau$ — the delay the bond imposes
- $\Phi$ — the phase relation the branch requires in order to close

That renaming is not decoration. It exposes an assumption that was buried in the word "stage": **every
crossing so far has been first-order.** A first-order lag can only attenuate — its magnitude is
$1/\sqrt{1+(\omega/\gamma)^2}$, always below 1 — and it supplies at most $\pi/2$ of phase, reached only
asymptotically where it attenuates to nothing.

But a crossing need not be first-order. A crossing with inertia — a resonator — has a phase that sweeps a
full $\pi$, and a magnitude that **peaks above 1** at its resonance. If crossings can differ in kind, then
"how many crossings" is the wrong accounting, and the bill has a price that depends on what kind of bond you
build.

Two questions, both falsifiable, and I got the first one wrong before running it:

1. **Is the dimension bound about stage count or about total dynamical order?** I predicted last time that a
   single second-order stage should oscillate, since it supplies up to $\pi$ alone. That is a clean
   prediction and it is **wrong** — the test below shows why.
2. **Does the kind of crossing change the gain bill?** If a resonant crossing amplifies while it phase-shifts,
   the external gain required should fall. By how much is a number.

---

## Step 1 — Library

The loop is negative feedback around a cascade of crossings, each with DC gain 1, times a loop gain $G$:

$$1 + G\prod_k H_k(s) = 0$$

Hopf when $\sum_k \angle H_k(i\omega) = \pi$ and $G\prod_k |H_k(i\omega)| = 1$. Two crossing types:

$$H^{(1)}(s) = \frac{1}{1+s/\gamma} \qquad H^{(2)}(s) = \frac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$


In [1]:
import numpy as np
from scipy.optimize import brentq
from scipy.integrate import solve_ivp

# ===========================================================================
# A crossing is ('1', gamma) for a first-order lag, or ('2', (wn, zeta)) for a
# resonant second-order stage. Both have DC gain 1, so G is the whole loop gain.
# ===========================================================================
def angle_and_mag(stages, w):
    """Total phase lag and total magnitude of the cascade at frequency w."""
    ph = 0.0; mg = 1.0
    for kind, p in stages:
        if kind == '1':
            gam = p
            ph += np.arctan2(w / gam, 1.0)
            mg /= np.hypot(1.0, w / gam)
        else:
            wn, z = p; r = w / wn
            ph += np.arctan2(2 * z * r, 1.0 - r * r) % (2 * np.pi)
            mg /= np.hypot(1.0 - r * r, 2 * z * r)
    return ph, mg

def hopf(stages, tau=0.0, Phi=np.pi):
    """Phase closure fixes w*; the magnitude condition then fixes G_crit."""
    F = lambda w: angle_and_mag(stages, w)[0] + w * tau - Phi
    if F(1e9) < 0:
        return None, None                      # the phase bill cannot be paid at any frequency
    hi = 1.0
    while F(hi) < 0:
        hi *= 2
    w = brentq(F, 1e-9, hi, xtol=1e-15, rtol=1e-15)
    return 1.0 / angle_and_mag(stages, w)[1], w


# ===========================================================================
# Ground truth: assemble the closed-loop state matrix and read its eigenvalues.
# Nothing about the phase/magnitude argument enters here.
# ===========================================================================
def state_matrix(stages, G):
    idx = []; n = 0
    for kind, p in stages:
        idx.append(n); n += 1 if kind == '1' else 2
    A = np.zeros((n, n))
    for j, (kind, p) in enumerate(stages):
        i = idx[j]
        src = idx[j - 1] if j > 0 else idx[-1]     # previous crossing's output
        coef = 1.0 if j > 0 else -G                # the loop closes with -G
        if kind == '1':
            gam = p
            A[i, i] = -gam; A[i, src] += gam * coef
        else:
            wn, z = p
            A[i, i + 1] = 1.0
            A[i + 1, i] = -wn * wn; A[i + 1, i + 1] = -2 * z * wn
            A[i + 1, src] += wn * wn * coef
    return A

def Gcrit_spectral(stages, lo=1e-8, hi=1e9):
    """Bisect on where max Re eigenvalue crosses zero."""
    f = lambda G: np.max(np.linalg.eigvals(state_matrix(stages, G)).real)
    if f(hi) < 0:
        return None
    if f(lo) > 0:
        return lo
    for _ in range(200):
        m = np.sqrt(lo * hi)
        if f(m) > 0: hi = m
        else:        lo = m
    return hi


# ===========================================================================
# Nonlinear loop for confirmation: same structure, with tanh saturation so an
# unstable state settles into a bounded cycle instead of diverging.
#   y1' = -y1 + tanh(-G y2)
#   y2'' + 2 z y2' + y2 = y1
# ===========================================================================
def nonlinear_swing(z, G, T=3000.0):
    def rhs(t, s):
        y1, y2, v2 = s
        return [-y1 + np.tanh(-G * y2), v2, -2 * z * v2 - y2 + y1]
    s = solve_ivp(rhs, [0, T], [0.02, 0.0, 0.0], rtol=1e-11, atol=1e-13, dense_output=True)
    tl = s.sol(np.linspace(0.9 * T, T, 4000))
    return float(tl[1].max() - tl[1].min())

print("library ready")


library ready


---
## Step 2 — Order, not stage count

A strictly proper transfer function of total order $p$ has phase approaching $p\pi/2$ as $\omega\to\infty$,
where its magnitude falls as $\omega^{-p}$. So:

- $p \le 2$: reaching $\pi$ needs $\omega\to\infty$, where the magnitude vanishes and $G\to\infty$. **No finite
  Hopf point exists.**
- $p \ge 3$: $\pi$ is reached at finite $\omega$ where the magnitude is finite. A threshold exists.

That predicts the bound is on **total order**, not on how the order is packaged. A single second-order stage
has order 2 and should therefore fail — *even with strong resonance* — which contradicts what I predicted last
time.


In [2]:
cases = [
 ("1 x first-order                 (order 1)", [('1', 1.0)]),
 ("2 x first-order                 (order 2)", [('1', 1.0), ('1', 1.0)]),
 ("1 x second-order  zeta=1.0      (order 2)", [('2', (1.0, 1.0))]),
 ("1 x second-order  zeta=0.1      (order 2)", [('2', (1.0, 0.1))]),
 ("1 x second-order  zeta=0.01     (order 2)", [('2', (1.0, 0.01))]),
 ("3 x first-order                 (order 3)", [('1', 1.0)] * 3),
 ("1st + 2nd  zeta=1.0             (order 3)", [('1', 1.0), ('2', (1.0, 1.0))]),
 ("1st + 2nd  zeta=0.3             (order 3)", [('1', 1.0), ('2', (1.0, 0.3))]),
 ("1st + 2nd  zeta=0.05            (order 3)", [('1', 1.0), ('2', (1.0, 0.05))]),
 ("1st + 2nd  zeta=0.01            (order 3)", [('1', 1.0), ('2', (1.0, 0.01))]),
 ("2 x second-order  zeta=0.1      (order 4)", [('2', (1.0, 0.1))] * 2),
]
print("G_crit from phase closure, against G_crit from the eigenvalues of the state matrix.\n")
print(f"{'arrangement':<44} {'phase closure':>14} {'spectral':>12} {'w*':>9}")
print("-" * 84)
for name, st in cases:
    ga, w = hopf(st); gn = Gcrit_spectral(st)
    sa = "none" if ga is None else f"{ga:.6f}"
    sn = "none" if gn is None else f"{gn:.6f}"
    sw = "   -   " if w is None else f"{w:.4f}"
    print(f"{name:<44} {sa:>14} {sn:>12} {sw:>9}")
print("""
   Every order-2 arrangement: no threshold at any gain up to 1e9 -- including a
   resonator at zeta=0.01. Resonance does NOT rescue order 2.
   Every order-3 arrangement: a threshold exists, and the two independent
   calculations agree to six digits.
""")


G_crit from phase closure, against G_crit from the eigenvalues of the state matrix.

arrangement                                   phase closure     spectral        w*
------------------------------------------------------------------------------------
1 x first-order                 (order 1)              none         none      -   
2 x first-order                 (order 2)              none         none      -   
1 x second-order  zeta=1.0      (order 2)              none         none      -   
1 x second-order  zeta=0.1      (order 2)              none         none      -   
1 x second-order  zeta=0.01     (order 2)              none         none      -   
3 x first-order                 (order 3)          8.000000     8.000000    1.7321
1st + 2nd  zeta=1.0             (order 3)          8.000000     8.000000    1.7321
1st + 2nd  zeta=0.3             (order 3)          1.560000     1.560000    1.2649
1st + 2nd  zeta=0.05            (order 3)          0.210000     0.210000    1.0488


**Prediction 1 confirmed, and my earlier guess refuted.** A single second-order stage cannot oscillate at
any gain, at any damping. The bound is on total dynamical order, $p\ge3$, and packaging is irrelevant to
*whether* the bill can be paid.

But look at the order-3 rows. The threshold spans $8.0$ down to $0.0404$ — a factor of **198** — at identical
total order. So packaging is entirely relevant to *what the bill costs.*

And note the row that is its own consistency check: `1st + 2nd zeta=1.0` returns exactly $8.000000$, the same
as three first-order stages. A critically damped second-order stage **is** two coincident real poles, so those
two arrangements are the same system. Nothing was fitted to make that happen.

---
## Step 3 — The price of a resonant crossing

Solve the order-3 case in closed form. With $\gamma=\omega_n=1$ and damping $\zeta$, phase closure gives

$$\arctan\omega + \arctan\!\frac{2\zeta\omega}{1-\omega^2} = \pi \quad\Longrightarrow\quad \boxed{\omega^* = \sqrt{1+2\zeta}}$$

and substituting into the magnitude condition,

$$G_{\text{crit}} = \sqrt{1+\omega^{*2}}\cdot\sqrt{(1-\omega^{*2})^2 + (2\zeta\omega^*)^2} = \boxed{4\zeta(1+\zeta)}$$

In terms of the resonator's quality factor $Q = 1/(2\zeta)$:

$$G_{\text{crit}} = \frac{2}{Q} + \frac{1}{Q^2}$$

Two things to check. Does the closed form hold? And at $\zeta=1$ it must return $4(1)(2)=8$ — the cascade
value — or it is wrong.


In [3]:
print("Closed forms:   w* = sqrt(1+2z)      G_crit = 4z(1+z) = 2/Q + 1/Q^2\n")
print(f"{'zeta':>7} {'Q':>7} | {'w* solved':>12} {'sqrt(1+2z)':>12} | {'G_crit solved':>14} {'4z(1+z)':>11} {'2/Q':>9}")
print("-" * 82)
for z in [1.0, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.001]:
    G, w = hopf([('1', 1.0), ('2', (1.0, z))])
    Q = 1 / (2 * z)
    print(f"{z:>7} {Q:>7.1f} | {w:>12.8f} {np.sqrt(1+2*z):>12.8f} | {G:>14.8f} {4*z*(1+z):>11.6f} {2/Q:>9.4f}")
print("\n   At zeta=1: G_crit = 8.0 exactly -- the three-first-order cascade value.")
print("   Critical damping IS two coincident real poles, so the formula must and does interpolate.")

print("\n\nWho actually pays, at the operating frequency w*:")
print(f"{'zeta':>7} {'Q':>7} {'w*':>9} {'|H1|':>8} {'|H2|':>10} {'product':>10} {'G_crit':>10} {'round trip':>12}")
print("-" * 80)
for z in [0.5, 0.2, 0.1, 0.03, 0.01]:
    G, w = hopf([('1', 1.0), ('2', (1.0, z))])
    h1 = 1 / np.hypot(1.0, w)
    h2 = 1 / np.hypot(1 - w*w, 2*z*w)
    print(f"{z:>7} {1/(2*z):>7.1f} {w:>9.5f} {h1:>8.4f} {h2:>10.4f} {h1*h2:>10.4f} {G:>10.5f} {G*h1*h2:>12.8f}")
print("""
   The round-trip gain is exactly 1 at threshold, always -- that condition is never bent.
   What changes is WHO supplies it. The resonant crossing has |H2| up to 35, so it
   amplifies while it phase-shifts, and the external gain only covers the remainder.
   A first-order lag has |H| < 1 always: it can never do this.
""")


Closed forms:   w* = sqrt(1+2z)      G_crit = 4z(1+z) = 2/Q + 1/Q^2

   zeta       Q |    w* solved   sqrt(1+2z) |  G_crit solved     4z(1+z)       2/Q
----------------------------------------------------------------------------------
    1.0     0.5 |   1.73205081   1.73205081 |     8.00000000    8.000000    4.0000
    0.5     1.0 |   1.41421356   1.41421356 |     3.00000000    3.000000    2.0000
    0.3     1.7 |   1.26491106   1.26491106 |     1.56000000    1.560000    1.2000
    0.2     2.5 |   1.18321596   1.18321596 |     0.96000000    0.960000    0.8000
    0.1     5.0 |   1.09544512   1.09544512 |     0.44000000    0.440000    0.4000
   0.05    10.0 |   1.04880885   1.04880885 |     0.21000000    0.210000    0.2000
   0.02    25.0 |   1.01980390   1.01980390 |     0.08160000    0.081600    0.0800
   0.01    50.0 |   1.00995049   1.00995049 |     0.04040000    0.040400    0.0400
  0.005   100.0 |   1.00498756   1.00498756 |     0.02010000    0.020100    0.0200
  0.001   500.0 | 

So the resonant crossing is not "a cheaper phase currency." It is a crossing that **pays both bills at
once** — phase and gain — because its magnitude peaks above unity where its phase is useful. The unity
round-trip condition is untouched; the resonator simply covers most of it.

That also settles a limit that delay cannot reach.


In [4]:
print("A 2 x first-order loop cannot clock at any gain. Two different ways to fix it:\n")
print("  add delay (order stays 2):")
for tau in [0.3, 0.6046, 1.0, 2.0, 5.0, 20.0, 100.0]:
    G, w = hopf([('1', 1.0), ('1', 1.0)], tau=tau)
    print(f"     tau={tau:<7} G_crit={G:>9.4f}   w*={w:.4f}")
print("\n  make one crossing resonant (order becomes 3):")
for z in [0.3, 0.1, 0.03, 0.01, 0.003]:
    G, w = hopf([('1', 1.0), ('2', (1.0, z))])
    print(f"     zeta={z:<6} Q={1/(2*z):<6.1f} G_crit={G:>9.4f}   w*={w:.4f}")
print("""
   Delay drives G_crit down toward 1 and stops: lags can only attenuate, so the
   external gain must still supply the whole unity round trip.
   Resonance drives G_crit toward 0 without bound, because the resonator itself
   supplies the gain. Only a resonant bond breaks the unity-external-gain floor.
""")


A 2 x first-order loop cannot clock at any gain. Two different ways to fix it:

  add delay (order stays 2):
     tau=0.3     G_crit=   7.3463   w*=2.5192
     tau=0.6046  G_crit=   4.0000   w*=1.7321
     tau=1.0     G_crit=   2.7071   w*=1.3065
     tau=2.0     G_crit=   1.7402   w*=0.8603
     tau=5.0     G_crit=   1.2087   w*=0.4569
     tau=20.0    G_crit=   1.0204   w*=0.1429
     tau=100.0   G_crit=   1.0009   w*=0.0308

  make one crossing resonant (order becomes 3):
     zeta=0.3    Q=1.7    G_crit=   1.5600   w*=1.2649
     zeta=0.1    Q=5.0    G_crit=   0.4400   w*=1.0954
     zeta=0.03   Q=16.7   G_crit=   0.1236   w*=1.0296
     zeta=0.01   Q=50.0   G_crit=   0.0404   w*=1.0100
     zeta=0.003  Q=166.7  G_crit=   0.0120   w*=1.0030

   Delay drives G_crit down toward 1 and stops: lags can only attenuate, so the
   external gain must still supply the whole unity round trip.
   Resonance drives G_crit toward 0 without bound, because the resonator itself
   supplies the gain.

$G_{\text{crit}}\to1$ for delay, $G_{\text{crit}}\to0$ for resonance. That is a qualitative difference between
the two, not just a rate difference, and it has a one-line reason: a lag can only lose signal, a resonator can
make it.

---
## Step 4 — Nonlinear confirmation

Linear thresholds are exact statements about where a steady state loses stability. Confirm they govern the
nonlinear system: same loop, `tanh` saturation, and check for a bounded cycle above and decay below.


In [5]:
print("Nonlinear loop with tanh saturation. Peak-to-peak of y2 in the last 10% of the run.\n")
print(f"{'zeta':>7} {'G_crit':>9} | {'at 0.7 G_crit':>26} | {'at 1.6 G_crit':>24}")
print("-" * 70)
ok = tot = 0
for z in [0.3, 0.1, 0.05]:
    G, _ = hopf([('1', 1.0), ('2', (1.0, z))])
    lo = nonlinear_swing(z, 0.7 * G); hi = nonlinear_swing(z, 1.6 * G)
    good = lo < 1e-4 and hi > 1e-3; ok += good; tot += 1
    print(f"{z:>7} {G:>9.5f} | G={0.7*G:<7.4f} swing={lo:>9.2e} | G={1.6*G:<7.4f} swing={hi:>8.2e}"
          f"  {'ok' if good else 'MISS'}")
print(f"\n   {ok} of {tot} agree.")

print("\nThe low-damping cases need a longer run -- at zeta=0.02 the decay rate is tiny,")
print("so a short record shows a transient that looks like a cycle:")
G, _ = hopf([('1', 1.0), ('2', (1.0, 0.02))])
for T in [600, 2000, 6000]:
    print(f"   zeta=0.02, G=0.7 G_crit={0.7*G:.5f}, T={T:<6} swing={nonlinear_swing(0.02, 0.7*G, T):.3e}")
print("   -> a slow transient, not a cycle. Run length has to scale with proximity to threshold.")

print("\nPushed to very high Q, with runs long enough to settle the question:")
for z in [0.01, 0.005]:
    G, _ = hopf([('1', 1.0), ('2', (1.0, z))])
    lo = nonlinear_swing(z, 0.7 * G, T=8000.0); hi = nonlinear_swing(z, 1.6 * G, T=3000.0)
    print(f"   zeta={z:<6} Q={1/(2*z):<5.0f} G_crit={G:.6f} (4z(1+z)={4*z*(1+z):.6f})"
          f"   below={lo:.2e}   above={hi:.2e}")
print("\n   The law governs the nonlinear system at external gains as low as 0.02.")


Nonlinear loop with tanh saturation. Peak-to-peak of y2 in the last 10% of the run.

   zeta    G_crit |              at 0.7 G_crit |            at 1.6 G_crit
----------------------------------------------------------------------
    0.3   1.56000 | G=1.0920  swing= 2.08e-12 | G=2.4960  swing=1.36e+00  ok
    0.1   0.44000 | G=0.3080  swing= 4.56e-12 | G=0.7040  swing=4.83e+00  ok
   0.05   0.21000 | G=0.1470  swing= 7.63e-12 | G=0.3360  swing=1.01e+01  ok

   3 of 3 agree.

The low-damping cases need a longer run -- at zeta=0.02 the decay rate is tiny,
so a short record shows a transient that looks like a cycle:
   zeta=0.02, G=0.7 G_crit=0.05712, T=600    swing=1.152e-03
   zeta=0.02, G=0.7 G_crit=0.05712, T=2000   swing=7.333e-07
   zeta=0.02, G=0.7 G_crit=0.05712, T=6000   swing=1.486e-11
   -> a slow transient, not a cycle. Run length has to scale with proximity to threshold.

Pushed to very high Q, with runs long enough to settle the question:
   zeta=0.01   Q=50    G_crit=0.0404

---
## Step 5 — The transfer check

Both previous notebooks were tested by whether their law showed up unchanged in a substrate that had no part
in deriving it. This one has an unusually direct check, because $G_{\text{crit}} = 2/Q$ is the reason precision
clocks are built the way they are.

Every high-accuracy oscillator in existence is **a high-$Q$ resonator plus a small amplifier**, not a long
gain chain:

| Resonator | $Q$ | Required external gain $\approx 2/Q$ |
|---|---|---|
| Pendulum in air | $10^2$–$10^3$ | $10^{-2}$–$10^{-3}$ |
| LC tank | $10^2$ | $\sim2\times10^{-2}$ |
| Quartz crystal | $10^4$–$10^6$ | $10^{-4}$–$10^{-6}$ |
| Atomic transition | $10^{8}$+ | $\lesssim10^{-8}$ |

A quartz oscillator runs on a sustaining amplifier that contributes almost nothing, which is exactly what
$2/Q\sim10^{-5}$ says it needs. Meanwhile the ring oscillator from the second notebook — three inverting
crossings, no resonance — needs per-stage gain above $\sec(\pi/3)=2$, and that is the design rule every
circuit designer knows. **Same equation, opposite corners, and both corners are real engineering practice.**

The choice between them is a trade the phase law makes visible: cascades are cheap to build and coarse;
resonators are expensive to build and nearly free to drive. Nothing in the framework had to be told that.

---
## Step 6 — What this does to the resource table

The previous notebook's table listed stages and delay as the interchangeable phase currencies. That was
right as far as it went and it was incomplete, because it assumed all crossings were the same kind.

| Resource | What it supplies | Limit it can reach |
|---|---|---|
| Total order $p$ | whether $\pi$ is reachable at finite $\omega$ at all | $p\ge3$ required — **no** amount of anything else substitutes |
| Stages (at fixed kind) | $\arccos(1/G)$ of phase each | exchanges with delay at $\arccos(1/G)/\sqrt{G^2-1}$ |
| Delay $\tau$ | $\tau\sqrt{G^2-1}$ of phase, unbounded | drives $G_{\text{crit}}\to1$, never below |
| **Resonance ($Q$)** | **phase and gain together** | drives $G_{\text{crit}}\to0$ — the only currency that does |
| Mass $M$ | feasibility of separable maintained basins | a floor, not a phase term |
| Noise | amplitude on credit | buys no coherence (previous notebook) |

The restructuring is that the resources split into two genuinely different kinds:

**A gate**, which nothing substitutes for: total order $\ge3$. Below it the bill is unpayable at any price.

**A price**, which many things pay, at published rates: $\sec(\pi/N)$ for cascades, the delay exchange rate,
$2/Q$ for resonance.

That distinction was invisible while every crossing was first-order.

---
## Step 7 — The access reading, with a number attached

The access-geometry framing says a method exists for a reader only when some path through its interface can
pay that method's access cost, and that learning is a **decrease in access distance** to something the
substrate could always do.

$G_{\text{crit}} = 2/Q$ is the first number on that claim. It says the access cost of the clock method is set
by **the quality of the bond, not the number of crossings.** Two readers at the same total order, the same
substrate, the same verb, differ by a factor of 198 in what they must supply — entirely because of how their
crossings are built.

So the sharpened statement:

$$\boxed{\;\text{Depth is not more crossings. Depth is crossings that return more of what passed through them.}\;}$$

And the limit case says the same thing more starkly. As $Q\to\infty$ the external gain goes to zero: the loop
becomes a resonator that merely needs its losses replaced. The reader stops driving the transformation and
starts only maintaining it. That is the cheapest possible access to the clock verb, and it is what a pendulum
clock is: an escapement returning the energy that friction removed, nothing more.

---
## Step 8 — Scoring

| Prediction | Result | Number |
|---|---|---|
| Bound is on total order, not stage count | **confirmed** | no threshold for any order-2 arrangement up to $G=10^9$ |
| A single second-order stage oscillates | **refuted** (my prediction, last notebook) | order 2 fails even at $\zeta=0.01$ |
| Crossing kind changes the gain bill | **confirmed** | $8.0\to0.0404$ at fixed order, a factor of 198 |
| Closed form for the resonant case | **derived and verified** | $\omega^*=\sqrt{1+2\zeta}$, $G_{\text{crit}}=4\zeta(1+\zeta)$, 10 of 10 |
| Must reproduce the cascade at $\zeta=1$ | **confirmed, unfitted** | returns exactly $8.000000$ |
| Round-trip gain still exactly 1 | **confirmed** | $1.00000000$ at every $\zeta$ tested |
| Resonance breaks the floor delay cannot | **confirmed** | delay $\to1$, resonance $\to0$ |
| Linear threshold governs the nonlinear loop | **confirmed** | down to external gain $0.02$ |

**What is not new.** None of the mathematics. This is the Barkhausen criterion, the Nyquist plot, and the
standard result that oscillator phase noise improves with resonator $Q$. Any RF engineer would read Step 3 as
routine. $G_{\text{crit}}=2/Q$ is why crystal oscillators are designed the way they are, and has been known for
a century.

**What the exercise established.** The framing question was whether the verb-level requirement keeps selecting
the right object as the substrate changes. Here the substrate changed in a way that broke one of my own
predictions, and the law absorbed the break by splitting one resource concept into two — a gate on order and a
price on placement — rather than acquiring a patch. That is the behavior of a correct object.

**One error worth recording.** The $\zeta=0.02$ below-threshold case read a swing of $1.2\times10^{-3}$ at
$T=600$ and would have been logged as a violation. At $T=6000$ it is $1.5\times10^{-11}$. This is the third
notebook in a row where a near-threshold slow transient masqueraded as a cycle. It is not an occasional
nuisance: **as any bifurcation is approached the decay rate goes to zero by construction**, so run length must
scale with proximity to the boundary. That belongs in the method, not the errata.

---
## Step 9 — Still open

- **Spatial extent.** Transport supplies phase like delay but with dispersion, and a diffusive crossing is
  neither a lag nor a resonator. Whether a reaction–diffusion ring obeys the same closure with an effective
  transport term, and where it sits between $G_{\text{crit}}\to1$ and $\to0$, is untested and is now the
  sharpest remaining question.
- **Non-monotone and non-Hill transfer shapes.** The law only ever referenced $f'$ at the operating point, so
  it should be shape-blind. Still an assertion.
- **Global structure.** Every threshold here is local. The coexisting latch-and-cycle case found in the second
  notebook says the local law does not enumerate attractors, and nothing since has addressed basins.
- **The Copy verb.** Untouched across four notebooks, and still the one place the verb set is known to change
  with depth rather than merely reprice: classical fan-out is free because many carriers share a signal, and
  no-cloning forbids it for a single quantum. Whether any other verb has a hard boundary rather than a price
  is the question the whole programme has been circling.
